In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from hydra import compose, initialize

from main.model.script.hydra_beans import KdConfig

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

To evaluate the good contribution of modalities we see MRR of it with and without the metric <br>
So if I have EEG, Aud, Txt and Vid and decide to ablate *vid* I measure:

A:MRR_mean modalities model that can use Vid but without video in inputstream <br>
B:MRR_mean across modalities of the ablated model

If B > A → Video is likely hurting other modalities<br>
If B < A → Video is likely helping them<br>
If B ~ A → Video has little effect on them<br>

In [ ]:
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

# TODO change?
baseline_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/epochepoch=45-stepstep=117484.ckpt"
ckpt = get_model_ckpt(baseline_checkpoint_path)
baseline = Factory.best_inference().build()
baseline.load_state_dict(ckpt, strict=False)
baseline.eval()

In [ ]:
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

trainer = lightning.Trainer(precision="16-mixed")

# Audio

In [ ]:
from main.core_data.media.audio import Audio

audio_less_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-aud/2026-03-22_22-14-48/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Audio.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_audioless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_audioless_results = trainer.validate(baseline_audioless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

Main effect of audio: maybe positive, neutral, or slightly negative

# Txt

In [ ]:
from main.core_data.media.text import Text

checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-22_11-24-33/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set([Text.modality_code()])).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

Text seems to fluctuate a lot from seed to seed. <br>
The modality is unstable.

> What if the increase in performance by removing audio is because it mitigates the presence of txt and thus removing modaltiies while txt is in it always proves gains?

# ECG

In [ ]:
from main.core_data.media.text import Text

audio_less_checkpoint_path = ""

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Text.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

full_datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baseline.pivot.code] + baseline.fusion_keys()
)

baseline_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=full_datamodule)
baseline_modless_module = EasyEegAviKdVateMaskedModule(baseline, None, datamodule=datamodule, )
audio_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )

baseline_results = trainer.validate(baseline_module, datamodule=full_datamodule)
baseline_modless_results = trainer.validate(baseline_modless_module, datamodule=datamodule)
audio_less_results = trainer.validate(audio_less_module, datamodule=datamodule)

In [ ]:
baseline_results

In [ ]:
audio_less_results

# MoCo less

In [1]:
from hydra import compose, initialize
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

from main.model.script.hydra_beans import KdConfig
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

trainer = lightning.Trainer(precision="16-mixed")

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'train-local.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Baseline model load

In [3]:
baselines = []
seed_ckpt = [
    # Seed=42. For some reason this are broken. (Model architecture mismatch on keys?)
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-19_15-32-23/checkpoints/last.ckpt",
    # Seed=150
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-24_00-32-18/checkpoints/epochepoch=38-stepstep=99567.ckpt",
    # Seed=1
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-26_11-25-28/checkpoints/epochepoch=36-stepstep=94461.ckpt",
]

for path in seed_ckpt:
    ckpt = get_model_ckpt(path)
    baseline = Factory.best_inference().build()
    # Load state of the seed ckpt
    baseline.load_state_dict(ckpt, strict=False)
    baseline.eval()
    # Append the built model
    baselines.append(baseline)

In [4]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[baselines[0].pivot.code] + baselines[0].fusion_keys()
)

In [5]:
baseline_res = []
for b in baselines:
    baseline = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_res.append(trainer.validate(baseline, datamodule=datamodule)[0])

You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.09319126605987549      │
│    val/fused/alignment_ecg    │     -0.07486465573310852      │
│    val/fused/alignment_eeg    │      0.1790904998779297       │
│    val/fused/alignment_txt    │     -0.06974675506353378      │
│    val/fused/alignment_vid    │      0.12579084932804108      │
│     val/fused/margin_aud      │      0.2772243618965149       │
│     val/fused/margin_ecg      │      0.16955387592315674      │
│     val/fused/margin_eeg      │      0.3371985852718353       │
│     val/fused/margin_txt      │      0.09508460015058517      │
│     val/fused/margin_vid      │      0.39437222480773926      │
│ val/fused/meanR@1-3-5-10_aud  │      0.8843636512756348       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.21155959367752075      │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9860917329788208       │
│ val/fused/meanR@1-3-5-10_mean │      0.5818352699279785       │
│ val/fused/meanR@1-3-5-10_txt  │     0.029801076278090477      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7973601818084717       │
│       val/fused/mrr_aud       │      0.8599337935447693       │
│       val/fused/mrr_ecg       │      0.1713300198316574       │
│       val/fused/mrr_eeg       │      0.9800578951835632       │
│      val/fused/mrr_mean       │      0.5491713881492615       │
│       val/fused/mrr_txt       │     0.023113396018743515      │
│       val/fused/mrr_vid       │      0.7114218473434448       │
│      val/fused/top10_aud      │      0.9304202795028687       │
│      val/fused/top10_ecg      │      0.33141130208969116      │
│      val/fused/top10_eeg      │      0.9949050545692444       │
│     val/fused/top10_mean      │      0.6453568339347839       │
│      val/fused/top10_txt      │      0.04845946654677391      │
│      val/fused/top10_vid      │      0.9215880036354065       │
│      val/fused/top1_aud       │      0.8211023807525635       │
│      val/fused/top1_ecg       │      0.0936298742890358       │
│      val/fused/top1_eeg       │       0.970745325088501       │
│      val/fused/top1_mean      │      0.4964584410190582       │
│      val/fused/top1_txt       │     0.010619204491376877      │
│      val/fused/top1_vid       │      0.5861955285072327       │
│      val/fused/top3_aud       │      0.8826127052307129       │
│      val/fused/top3_ecg       │      0.18478858470916748      │
│      val/fused/top3_eeg       │      0.9875913858413696       │
│      val/fused/top3_mean      │      0.5775818228721619       │
│      val/fused/top3_txt       │     0.024827998131513596      │
│      val/fused/top3_vid       │      0.8080882430076599       │
│      val/fused/top5_aud       │      0.9033191800117493       │
│      val/fused/top5_ecg       │      0.23640857636928558      │
│      val/fused/top5_eeg       │      0.9911249876022339       │
│      val/fused/top5_mean      │      0.6079438328742981       │
│      val/fused/top5_txt       │     0.035297635942697525      │
│      val/fused/top5_vid       │      0.8735688924789429       │
│        val/fusion-loss        │      3.7288026809692383       │
│        val/fusion/aud         │       3.478576898574829       │
│        val/fusion/ecg         │       5.91384220123291        │
│        val/fusion/eeg         │       2.42000412940979        │
│        val/fusion/txt         │       6.042628288269043       │
│        val/fusion/vid         │      3.1774497032165527       │
│           val/loss            │      3.7288026809692383       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.1320195198059082       │
│    val/fused/alignment_ecg    │      0.2077609896659851       │
│    val/fused/alignment_eeg    │      0.1400308459997177       │
│    val/fused/alignment_txt    │      0.08619217574596405      │
│    val/fused/alignment_vid    │      0.14835308492183685      │
│     val/fused/margin_aud      │      0.27854952216148376      │
│     val/fused/margin_ecg      │      0.4161883592605591       │
│     val/fused/margin_eeg      │      0.28491073846817017      │
│     val/fused/margin_txt      │      0.3013633191585541       │
│     val/fused/margin_vid      │      0.3650020360946655       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9567981958389282       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8625068664550781       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9617470502853394       │
│ val/fused/meanR@1-3-5-10_mean │      0.8180058598518372       │
│ val/fused/meanR@1-3-5-10_txt  │       0.546066403388977       │
│ val/fused/meanR@1-3-5-10_vid  │       0.762910783290863       │
│       val/fused/mrr_aud       │      0.9350941777229309       │
│       val/fused/mrr_ecg       │      0.7973181009292603       │
│       val/fused/mrr_eeg       │      0.9460970163345337       │
│      val/fused/mrr_mean       │      0.7744465470314026       │
│       val/fused/mrr_txt       │      0.5198565721511841       │
│       val/fused/mrr_vid       │      0.6738667488098145       │
│      val/fused/top10_aud      │      0.9876675009727478       │
│      val/fused/top10_ecg      │      0.9577155709266663       │
│      val/fused/top10_eeg      │       0.984961748123169       │
│     val/fused/top10_mean      │      0.8838645815849304       │
│      val/fused/top10_txt      │      0.5883936882019043       │
│      val/fused/top10_vid      │      0.9005847573280334       │
│      val/fused/top1_aud       │      0.9027101397514343       │
│      val/fused/top1_ecg       │      0.7031850814819336       │
│      val/fused/top1_eeg       │      0.9234941005706787       │
│      val/fused/top1_mean      │      0.7102217078208923       │
│      val/fused/top1_txt       │      0.4754711389541626       │
│      val/fused/top1_vid       │      0.5462482571601868       │
│      val/fused/top3_aud       │      0.9619367122650146       │
│      val/fused/top3_ecg       │      0.8704009056091309       │
│      val/fused/top3_eeg       │      0.9641712307929993       │
│      val/fused/top3_mean      │      0.8228382468223572       │
│      val/fused/top3_txt       │      0.5516003370285034       │
│      val/fused/top3_vid       │      0.7660818696022034       │
│      val/fused/top5_aud       │      0.9748782515525818       │
│      val/fused/top5_ecg       │      0.9187259674072266       │
│      val/fused/top5_eeg       │      0.9743610620498657       │
│      val/fused/top5_mean      │      0.8550988435745239       │
│      val/fused/top5_txt       │      0.5688005089759827       │
│      val/fused/top5_vid       │      0.8387282490730286       │
│        val/fusion-loss        │       2.943612575531006       │
│        val/fusion/aud         │      3.0037081241607666       │
│        val/fusion/ecg         │      2.1381113529205322       │
│        val/fusion/eeg         │      2.9312548637390137       │
│        val/fusion/txt         │       3.455925941467285       │
│        val/fusion/vid         │       2.921487331390381       │
│           val/loss            │       2.943612575531006       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.13244906067848206      │
│    val/fused/alignment_ecg    │      0.21093937754631042      │
│    val/fused/alignment_eeg    │      0.1391088217496872       │
│    val/fused/alignment_txt    │      0.08658074587583542      │
│    val/fused/alignment_vid    │      0.14989322423934937      │
│     val/fused/margin_aud      │      0.27632975578308105      │
│     val/fused/margin_ecg      │      0.4186972677707672       │
│     val/fused/margin_eeg      │      0.28083252906799316      │
│     val/fused/margin_txt      │      0.3143104314804077       │
│     val/fused/margin_vid      │      0.35408851504325867      │
│ val/fused/meanR@1-3-5-10_aud  │      0.9588536024093628       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8432866930961609       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9625893235206604       │
│ val/fused/meanR@1-3-5-10_mean │      0.8099706768989563       │
│ val/fused/meanR@1-3-5-10_txt  │       0.515891432762146       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7692323327064514       │
│       val/fused/mrr_aud       │      0.9371210336685181       │
│       val/fused/mrr_ecg       │      0.7774708867073059       │
│       val/fused/mrr_eeg       │       0.948738157749176       │
│      val/fused/mrr_mean       │      0.7641531825065613       │
│       val/fused/mrr_txt       │      0.4770139157772064       │
│       val/fused/mrr_vid       │      0.6804219484329224       │
│      val/fused/top10_aud      │      0.9899513125419617       │
│      val/fused/top10_ecg      │      0.9442614316940308       │
│      val/fused/top10_eeg      │      0.9844686985015869       │
│     val/fused/top10_mean      │       0.880527913570404       │
│      val/fused/top10_txt      │      0.5794196724891663       │
│      val/fused/top10_vid      │      0.9045383334159851       │
│      val/fused/top1_aud       │      0.9045372009277344       │
│      val/fused/top1_ecg       │       0.680944561958313       │
│      val/fused/top1_eeg       │      0.9286711812019348       │
│      val/fused/top1_mean      │      0.6962954998016357       │
│      val/fused/top1_txt       │      0.4132515788078308       │
│      val/fused/top1_vid       │      0.5540729761123657       │
│      val/fused/top3_aud       │       0.963916003704071       │
│      val/fused/top3_ecg       │      0.8498077988624573       │
│      val/fused/top3_eeg       │      0.9635137915611267       │
│      val/fused/top3_mean      │      0.8137070536613464       │
│      val/fused/top3_txt       │      0.5201914310455322       │
│      val/fused/top3_vid       │      0.7711061239242554       │
│      val/fused/top5_aud       │      0.9770097732543945       │
│      val/fused/top5_ecg       │      0.8981329202651978       │
│      val/fused/top5_eeg       │      0.9737036228179932       │
│      val/fused/top5_mean      │      0.8493523001670837       │
│      val/fused/top5_txt       │      0.5507029891014099       │
│      val/fused/top5_vid       │      0.8472118973731995       │
│        val/fusion-loss        │       2.958447217941284       │
│        val/fusion/aud         │       3.006216287612915       │
│        val/fusion/ecg         │      2.1115896701812744       │
│        val/fusion/eeg         │       2.945300579071045       │
│        val/fusion/txt         │      3.4539785385131836       │
│        val/fusion/vid         │       2.928723096847534       │
│           val/loss            │       2.958447217941284       │
└───────────────────────────────┴───────────────────────────────┘

# Text

In [ ]:
mrr_check_keys = [
    'val/fused/mrr_ecg',
    'val/fused/mrr_aud',
    'val/fused/mrr_vid',
    # 'val/fused/mrr_txt', -> We are ablating text
    'val/fused/mrr_eeg',
    'val/fused/mrr_mean',
]

In [6]:
import numpy as np

# I only care for MRR + meanR@
print("Baseline variance calculations")

# Without text mean
baseline_res[0]["val/fused/mrr_mean"] = 0.8213
baseline_res[0]["val/fused/mrr_ecg"] = 0.6902
baseline_res[0]["val/fused/mrr_aud"] = 0.9383
baseline_res[0]["val/fused/mrr_eeg"] = 0.9641
baseline_res[0]["val/fused/mrr_vid"] = 0.6925
baseline_res[0]["val/fused/mrr_txt"] = 0.5746

for bb in baseline_res:
    bb["val/fused/mrr_mean"] = (
            (bb["val/fused/mrr_vid"] + bb["val/fused/mrr_aud"] + bb["val/fused/mrr_ecg"] + bb["val/fused/mrr_eeg"]) / 4
    )

baseline_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_res:
        values.append(res[key])

    arr = np.array(values)
    baseline_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_res)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

Baseline variance calculations
For key:val/fused/mrr_ecg on 3 baselines mean: 0.7549963292121887 std: 0.04652885260870428
For key:val/fused/mrr_aud on 3 baselines mean: 0.9368384037971497 std: 0.0013239420559275343
For key:val/fused/mrr_vid on 3 baselines mean: 0.6822628990809122 std: 0.007717570297504635
For key:val/fused/mrr_eeg on 3 baselines mean: 0.9529783913612366 std: 0.007937738596031712
For key:val/fused/mrr_mean on 3 baselines mean: 0.8317690058628718 std: 0.0074724029126425374


In [7]:
txt_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-25_12-03-51/checkpoints/epochepoch=38-stepstep=99567.ckpt"

In [8]:
# Create model instance
inference_ckpt = get_model_ckpt(txt_ablate_ckpt_150)
mod_less_model = Factory.best_inference(disabled_supports={'txt'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

# And its according datamodule
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

# Now validate it
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

# I only care for MRR at the moment
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.15200521051883698      │
│    val/fused/alignment_ecg    │      0.22985462844371796      │
│    val/fused/alignment_eeg    │      0.14616021513938904      │
│    val/fused/alignment_vid    │      0.16776132583618164      │
│     val/fused/margin_aud      │      0.3100909888744354       │
│     val/fused/margin_ecg      │      0.44195011258125305      │
│     val/fused/margin_eeg      │      0.29747962951660156      │
│     val/fused/margin_vid      │      0.3982754945755005       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9744595289230347       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8644975423812866       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9718751907348633       │
│ val/fused/meanR@1-3-5-10_mean │       0.899932324886322       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7888970971107483       │
│       val/fused/mrr_aud       │      0.9584898948669434       │
│       val/fused/mrr_ecg       │      0.8040436506271362       │
│       val/fused/mrr_eeg       │       0.959257185459137       │
│      val/fused/mrr_mean       │      0.8556908965110779       │
│       val/fused/mrr_vid       │       0.700972855091095       │
│      val/fused/top10_aud      │      0.9948234558105469       │
│      val/fused/top10_ecg      │      0.9494783282279968       │
│      val/fused/top10_eeg      │      0.9905497431755066       │
│     val/fused/top10_mean      │      0.9633274078369141       │
│      val/fused/top10_vid      │       0.918458104133606       │
│      val/fused/top1_aud       │      0.9349878430366516       │
│      val/fused/top1_ecg       │      0.7158154845237732       │
│      val/fused/top1_eeg       │      0.9405866861343384       │
│      val/fused/top1_mean      │       0.791637122631073       │
│      val/fused/top1_vid       │      0.5751585364341736       │
│      val/fused/top3_aud       │      0.9794458150863647       │
│      val/fused/top3_ecg       │      0.8761669397354126       │
│      val/fused/top3_eeg       │      0.9737036228179932       │
│      val/fused/top3_mean      │      0.9067772626876831       │
│      val/fused/top3_vid       │      0.7977925539016724       │
│      val/fused/top5_aud       │      0.9885810613632202       │
│      val/fused/top5_ecg       │      0.9165294170379639       │
│      val/fused/top5_eeg       │      0.9826608300209045       │
│      val/fused/top5_mean      │      0.9379876255989075       │
│      val/fused/top5_vid       │      0.8641791939735413       │
│        val/fusion-loss        │      2.6421408653259277       │
│        val/fusion/aud         │       2.687180519104004       │
│        val/fusion/ecg         │      1.8907060623168945       │
│        val/fusion/eeg         │      2.8457019329071045       │
│        val/fusion/vid         │      2.6839771270751953       │
│           val/loss            │      2.6421408653259277       │
└───────────────────────────────┴───────────────────────────────┘

For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.8040436506271362
key: val/fused/mrr_ecg  has gain of: 0.0490473214149475 

ablated_res for key: val/fused/mrr_aud  is: 0.9584898948669434
key: val/fused/mrr_aud  has gain of: 0.021651491069793694 

ablated_res for key: val/fused/mrr_vid  is: 0.700972855091095
key: val/fused/mrr_vid  has gain of: 0.01870995601018277 

ablated_res for key: val/fused/mrr_eeg  is: 0.959257185459137
key: val/fused/mrr_eeg  has gain of: 0.006278794097900331 

ablated_res for key: val/fused/mrr_mean  is: 0.8556908965110779
key: val/fused/mrr_mean  has gain of: 0.023921890648206046 



## Baseline without mod

In [9]:
baseline_mod_less_results = []
for b in baselines[1:]:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

baseline_mod_less_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_mod_less_results:
        values.append(res[key])

    arr = np.array(values)
    baseline_mod_less_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_mod_less_results)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

# Compare now
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_mod_less_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.14956499636173248      │
│    val/fused/alignment_ecg    │      0.21219755709171295      │
│    val/fused/alignment_eeg    │      0.14407804608345032      │
│    val/fused/alignment_vid    │      0.15417495369911194      │
│     val/fused/margin_aud      │      0.31456637382507324      │
│     val/fused/margin_ecg      │      0.4251786768436432       │
│     val/fused/margin_eeg      │      0.2969921827316284       │
│     val/fused/margin_vid      │      0.3919265866279602       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9636876583099365       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8565348982810974       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9635343551635742       │
│ val/fused/meanR@1-3-5-10_mean │      0.8902395367622375       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7772011756896973       │
│       val/fused/mrr_aud       │      0.9446651935577393       │
│       val/fused/mrr_ecg       │      0.7911146879196167       │
│       val/fused/mrr_eeg       │      0.9483951330184937       │
│      val/fused/mrr_mean       │      0.8433588743209839       │
│       val/fused/mrr_vid       │      0.6892603039741516       │
│      val/fused/top10_aud      │      0.9911693334579468       │
│      val/fused/top10_ecg      │      0.9541460871696472       │
│      val/fused/top10_eeg      │       0.985947847366333       │
│     val/fused/top10_mean      │       0.96016526222229        │
│      val/fused/top10_vid      │      0.9093979001045227       │
│      val/fused/top1_aud       │      0.9165651798248291       │
│      val/fused/top1_ecg       │      0.6954969763755798       │
│      val/fused/top1_eeg       │      0.9257128238677979       │
│      val/fused/top1_mean      │      0.7753505706787109       │
│      val/fused/top1_vid       │      0.5636273622512817       │
│      val/fused/top3_aud       │      0.9666565656661987       │
│      val/fused/top3_ecg       │      0.8640856742858887       │
│      val/fused/top3_eeg       │      0.9667186737060547       │
│      val/fused/top3_mean      │      0.8948392868041992       │
│      val/fused/top3_vid       │      0.7818960547447205       │
│      val/fused/top5_aud       │      0.9803593754768372       │
│      val/fused/top5_ecg       │      0.9124107956886292       │
│      val/fused/top5_eeg       │      0.9757580161094666       │
│      val/fused/top5_mean      │      0.9306029081344604       │
│      val/fused/top5_vid       │      0.8538835048675537       │
│        val/fusion-loss        │      2.7206764221191406       │
│        val/fusion/aud         │       2.697739601135254       │
│        val/fusion/ecg         │       2.087677001953125       │
│        val/fusion/eeg         │       2.872695207595825       │
│        val/fusion/vid         │      2.8334057331085205       │
│           val/loss            │      2.7206764221191406       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.15202611684799194      │
│    val/fused/alignment_ecg    │      0.21180877089500427      │
│    val/fused/alignment_eeg    │      0.14787837862968445      │
│    val/fused/alignment_vid    │      0.1573319435119629       │
│     val/fused/margin_aud      │       0.317172110080719       │
│     val/fused/margin_ecg      │      0.42477673292160034      │
│     val/fused/margin_eeg      │      0.29920539259910583      │
│     val/fused/margin_vid      │      0.38477233052253723      │
│ val/fused/meanR@1-3-5-10_aud  │      0.9674558639526367       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8371087312698364       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9657737016677856       │
│ val/fused/meanR@1-3-5-10_mean │      0.8888410329818726       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7850258946418762       │
│       val/fused/mrr_aud       │      0.9496665596961975       │
│       val/fused/mrr_ecg       │      0.7692552804946899       │
│       val/fused/mrr_eeg       │      0.9526851177215576       │
│      val/fused/mrr_mean       │      0.8426817059516907       │
│       val/fused/mrr_vid       │      0.6991198658943176       │
│      val/fused/top10_aud      │      0.9933009147644043       │
│      val/fused/top10_ecg      │      0.9423393607139587       │
│      val/fused/top10_eeg      │      0.9865230917930603       │
│     val/fused/top10_mean      │       0.959125816822052       │
│      val/fused/top10_vid      │      0.9143397808074951       │
│      val/fused/top1_aud       │      0.9238733649253845       │
│      val/fused/top1_ecg       │      0.6707853078842163       │
│      val/fused/top1_eeg       │      0.9336017370223999       │
│      val/fused/top1_mean      │      0.7762871384620667       │
│      val/fused/top1_vid       │      0.5768882036209106       │
│      val/fused/top3_aud       │       0.970310628414154       │
│      val/fused/top3_ecg       │      0.8410214185714722       │
│      val/fused/top3_eeg       │      0.9667186737060547       │
│      val/fused/top3_mean      │      0.8914486765861511       │
│      val/fused/top3_vid       │      0.7877439856529236       │
│      val/fused/top5_aud       │      0.9823386669158936       │
│      val/fused/top5_ecg       │      0.8942888379096985       │
│      val/fused/top5_eeg       │      0.9762511253356934       │
│      val/fused/top5_mean      │      0.9285025596618652       │
│      val/fused/top5_vid       │      0.8611316680908203       │
│        val/fusion-loss        │       2.704986572265625       │
│        val/fusion/aud         │      2.6713500022888184       │
│        val/fusion/ecg         │      2.0988872051239014       │
│        val/fusion/eeg         │      2.8232908248901367       │
│        val/fusion/vid         │      2.8173866271972656       │
│           val/loss            │       2.704986572265625       │
└───────────────────────────────┴───────────────────────────────┘

For key:val/fused/mrr_ecg on 2 baselines mean: 0.7801849842071533 std: 0.010929703712463379
For key:val/fused/mrr_aud on 2 baselines mean: 0.9471658766269684 std: 0.002500683069229126
For key:val/fused/mrr_vid on 2 baselines mean: 0.6941900849342346 std: 0.004929780960083008
For key:val/fused/mrr_eeg on 2 baselines mean: 0.9505401253700256 std: 0.0021449923515319824
For key:val/fused/mrr_mean on 2 baselines mean: 0.8430202901363373 std: 0.00033858418464660645
For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.8040436506271362
key: val/fused/mrr_ecg  has gain of: 0.02385866641998291 

ablated_res for key: val/fused/mrr_aud  is: 0.9584898948669434
key: val/fused/mrr_aud  has gain of: 0.011324018239974976 

ablated_res for key: val/fused/mrr_vid  is: 0.700972855091095
key: val/fused/mrr_vid  has gain of: 0.0067827701568603516 

ablated_res for key: val/fused/mrr_eeg  is: 0.959257185459137
key: val/fused/mrr_eeg  has gain of: 0.008717060089111328 

ablated_res for key: val/fused/mr

> The model appears robust to the absence of text at inference. Training without text still yields a modest improvement on shared-modality retrieval, suggesting that text may slightly complicate alignment during training rather than being strictly required at test time.

> Although removing text slightly improved shared-modality retrieval, the gain was modest. Since the broader goal of the model is to learn richer multimodal representations, text was retained as a support modality despite this small alignment cost.

or

> In this setting, text was derived from speech rather than being an independent source of information. Given its small negative effect on shared-modality retrieval and the likely redundancy with audio, it was excluded from the final configuration.

# Audio

In [10]:
aud_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-aud/2026-03-25_22-37-18/checkpoints/epochepoch=36-stepstep=94461.ckpt"

In [11]:
mrr_check_keys = [
    'val/fused/mrr_ecg',
    # 'val/fused/mrr_aud',
    'val/fused/mrr_vid',
    'val/fused/mrr_txt',
    'val/fused/mrr_eeg',
    'val/fused/mrr_mean',
]

In [12]:
import numpy as np

# I only care for MRR + meanR@
print("Baseline variance calculations")

# Without text mean
baseline_res[0]["val/fused/mrr_mean"] = 0.8213
baseline_res[0]["val/fused/mrr_ecg"] = 0.6902
baseline_res[0]["val/fused/mrr_aud"] = 0.9383
baseline_res[0]["val/fused/mrr_eeg"] = 0.9641
baseline_res[0]["val/fused/mrr_vid"] = 0.6925
baseline_res[0]["val/fused/mrr_txt"] = 0.5746

for bb in baseline_res:
    bb["val/fused/mrr_mean"] = (
            (bb["val/fused/mrr_vid"] + bb["val/fused/mrr_txt"] + bb["val/fused/mrr_ecg"] + bb["val/fused/mrr_eeg"]) / 4
    )

baseline_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_res:
        values.append(res[key])

    arr = np.array(values)
    baseline_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_res)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

Baseline variance calculations
For key:val/fused/mrr_ecg on 3 baselines mean: 0.7549963292121887 std: 0.04652885260870428
For key:val/fused/mrr_vid on 3 baselines mean: 0.6822628990809122 std: 0.007717570297504635
For key:val/fused/mrr_txt on 3 baselines mean: 0.5238234959761302 std: 0.039937979601370535
For key:val/fused/mrr_eeg on 3 baselines mean: 0.9529783913612366 std: 0.007937738596031712
For key:val/fused/mrr_mean on 3 baselines mean: 0.7285152789076169 std: 0.005611683659970961


In [16]:
# Create model instance
inference_ckpt = get_model_ckpt(aud_ablate_ckpt_150)
mod_less_model = Factory.best_inference(disabled_supports={'aud'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

# And its according datamodule
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

# Now validate it
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

# I only care for MRR at the moment
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_ecg    │       0.21602763235569        │
│    val/fused/alignment_eeg    │      0.1472318023443222       │
│    val/fused/alignment_txt    │      0.07744433730840683      │
│    val/fused/alignment_vid    │      0.1448354870080948       │
│     val/fused/margin_ecg      │      0.43120095133781433      │
│     val/fused/margin_eeg      │      0.2920091152191162       │
│     val/fused/margin_txt      │      0.33587658405303955      │
│     val/fused/margin_vid      │      0.35625141859054565      │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8571527004241943       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9731900691986084       │
│ val/fused/meanR@1-3-5-10_mean │      0.7740991115570068       │
│ val/fused/meanR@1-3-5-10_txt  │      0.47935986518859863      │
│ val/fused/meanR@1-3-5-10_vid  │       0.786693811416626       │
│       val/fused/mrr_ecg       │      0.7967884540557861       │
│       val/fused/mrr_eeg       │      0.9623293280601501       │
│      val/fused/mrr_mean       │      0.7237718105316162       │
│       val/fused/mrr_txt       │      0.43402549624443054      │
│       val/fused/mrr_vid       │      0.7019438743591309       │
│      val/fused/top10_ecg      │      0.9516749382019043       │
│      val/fused/top10_eeg      │       0.989399254322052       │
│     val/fused/top10_mean      │      0.8514279127120972       │
│      val/fused/top10_txt      │       0.551450788974762       │
│      val/fused/top10_vid      │      0.9131866693496704       │
│      val/fused/top1_ecg       │      0.7100494503974915       │
│      val/fused/top1_eeg       │      0.9460103511810303       │
│      val/fused/top1_mean      │      0.6489852070808411       │
│      val/fused/top1_txt       │      0.36060425639152527      │
│      val/fused/top1_vid       │      0.5792768001556396       │
│      val/fused/top3_ecg       │       0.859967052936554       │
│      val/fused/top3_eeg       │      0.9752649664878845       │
│      val/fused/top3_mean      │      0.7789301872253418       │
│      val/fused/top3_txt       │      0.48788514733314514      │
│      val/fused/top3_vid       │      0.7926035523414612       │
│      val/fused/top5_ecg       │      0.9069193005561829       │
│      val/fused/top5_eeg       │      0.9820855855941772       │
│      val/fused/top5_mean      │      0.8170530796051025       │
│      val/fused/top5_txt       │      0.5174992680549622       │
│      val/fused/top5_vid       │      0.8617082238197327       │
│        val/fusion-loss        │       2.974700927734375       │
│        val/fusion/ecg         │      2.0425291061401367       │
│        val/fusion/eeg         │       2.834406614303589       │
│        val/fusion/txt         │      3.6289188861846924       │
│        val/fusion/vid         │       2.981623649597168       │
│           val/loss            │       2.974700927734375       │
└───────────────────────────────┴───────────────────────────────┘

For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.7967884540557861
key: val/fused/mrr_ecg  has gain of: 0.0417921248435974 

ablated_res for key: val/fused/mrr_vid  is: 0.7019438743591309
key: val/fused/mrr_vid  has gain of: 0.01968097527821866 

ablated_res for key: val/fused/mrr_txt  is: 0.43402549624443054
key: val/fused/mrr_txt  has gain of: -0.08979799973169966 

ablated_res for key: val/fused/mrr_eeg  is: 0.9623293280601501
key: val/fused/mrr_eeg  has gain of: 0.009350936698913515 

ablated_res for key: val/fused/mrr_mean  is: 0.7237718105316162
key: val/fused/mrr_mean  has gain of: -0.0047434683760007035 



## Baseline without mod

In [17]:
baseline_mod_less_results = []
for b in baselines[1:]:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

baseline_mod_less_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_mod_less_results:
        values.append(res[key])

    arr = np.array(values)
    baseline_mod_less_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_mod_less_results)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

# Compare now
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_mod_less_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_ecg    │      0.21136006712913513      │
│    val/fused/alignment_eeg    │      0.14554594457149506      │
│    val/fused/alignment_txt    │      0.0807185173034668       │
│    val/fused/alignment_vid    │      0.15247978270053864      │
│     val/fused/margin_ecg      │      0.4388371706008911       │
│     val/fused/margin_eeg      │       0.295120507478714       │
│     val/fused/margin_txt      │      0.3050808906555176       │
│     val/fused/margin_vid      │      0.3808381259441376       │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8642916083335876       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9659379720687866       │
│ val/fused/meanR@1-3-5-10_mean │      0.7842849493026733       │
│ val/fused/meanR@1-3-5-10_txt  │       0.528679370880127       │
│ val/fused/meanR@1-3-5-10_vid  │      0.7782307863235474       │
│       val/fused/mrr_ecg       │      0.8018373847007751       │
│       val/fused/mrr_eeg       │      0.9516498446464539       │
│      val/fused/mrr_mean       │       0.735806941986084       │
│       val/fused/mrr_txt       │      0.5000666975975037       │
│       val/fused/mrr_vid       │      0.6896739602088928       │
│      val/fused/top10_ecg      │       0.957440972328186       │
│      val/fused/top10_eeg      │      0.9865230917930603       │
│     val/fused/top10_mean      │      0.8587589263916016       │
│      val/fused/top10_txt      │      0.5791205763816833       │
│      val/fused/top10_vid      │      0.9119511842727661       │
│      val/fused/top1_ecg       │      0.7092257142066956       │
│      val/fused/top1_eeg       │      0.9305612444877625       │
│      val/fused/top1_mean      │      0.6635946035385132       │
│      val/fused/top1_txt       │      0.4515405297279358       │
│      val/fused/top1_vid       │      0.5630508065223694       │
│      val/fused/top3_ecg       │      0.8742449283599854       │
│      val/fused/top3_eeg       │      0.9685265421867371       │
│      val/fused/top3_mean      │      0.7896910309791565       │
│      val/fused/top3_txt       │      0.5317080616950989       │
│      val/fused/top3_vid       │      0.7842846512794495       │
│      val/fused/top5_ecg       │      0.9162548184394836       │
│      val/fused/top5_eeg       │      0.9781411290168762       │
│      val/fused/top5_mean      │      0.8250951766967773       │
│      val/fused/top5_txt       │      0.5523481965065002       │
│      val/fused/top5_vid       │      0.8536363840103149       │
│        val/fusion-loss        │      2.9196062088012695       │
│        val/fusion/ecg         │      2.0875794887542725       │
│        val/fusion/eeg         │      2.8550972938537598       │
│        val/fusion/txt         │      3.5404722690582275       │
│        val/fusion/vid         │      2.8606624603271484       │
│           val/loss            │      2.9196062088012695       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_ecg    │      0.21674135327339172      │
│    val/fused/alignment_eeg    │      0.1435401886701584       │
│    val/fused/alignment_txt    │      0.08119089901447296      │
│    val/fused/alignment_vid    │      0.15208706259727478      │
│     val/fused/margin_ecg      │      0.4435393214225769       │
│     val/fused/margin_eeg      │      0.2892876863479614       │
│     val/fused/margin_txt      │      0.3206326365470886       │
│     val/fused/margin_vid      │      0.36867472529411316      │
│ val/fused/meanR@1-3-5-10_ecg  │      0.8490527272224426       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9646642804145813       │
│ val/fused/meanR@1-3-5-10_mean │      0.7740822434425354       │
│ val/fused/meanR@1-3-5-10_txt  │      0.49943915009498596      │
│ val/fused/meanR@1-3-5-10_vid  │      0.7831727266311646       │
│       val/fused/mrr_ecg       │      0.7854580283164978       │
│       val/fused/mrr_eeg       │      0.9508765935897827       │
│      val/fused/mrr_mean       │      0.7220102548599243       │
│       val/fused/mrr_txt       │      0.45557108521461487      │
│       val/fused/mrr_vid       │      0.6961352229118347       │
│      val/fused/top10_ecg      │       0.947830855846405       │
│      val/fused/top10_eeg      │       0.986194372177124       │
│     val/fused/top10_mean      │      0.8552126288414001       │
│      val/fused/top10_txt      │      0.5738857388496399       │
│      val/fused/top10_vid      │      0.9129396080970764       │
│      val/fused/top1_ecg       │      0.6924766898155212       │
│      val/fused/top1_eeg       │      0.9308899641036987       │
│      val/fused/top1_mean      │      0.6457417011260986       │
│      val/fused/top1_txt       │      0.3870774805545807       │
│      val/fused/top1_vid       │      0.5725228190422058       │
│      val/fused/top3_ecg       │       0.857495903968811       │
│      val/fused/top3_eeg       │      0.9654860496520996       │
│      val/fused/top3_mean      │      0.7775595784187317       │
│      val/fused/top3_txt       │      0.5007478594779968       │
│      val/fused/top3_vid       │      0.7865085005760193       │
│      val/fused/top5_ecg       │      0.8984074592590332       │
│      val/fused/top5_eeg       │      0.9760867357254028       │
│      val/fused/top5_mean      │      0.8178148865699768       │
│      val/fused/top5_txt       │       0.536045491695404       │
│      val/fused/top5_vid       │      0.8607198596000671       │
│        val/fusion-loss        │       2.94278883934021        │
│        val/fusion/ecg         │      2.0360569953918457       │
│        val/fusion/eeg         │      2.8838419914245605       │
│        val/fusion/txt         │       3.544590950012207       │
│        val/fusion/vid         │       2.888561248779297       │
│           val/loss            │       2.94278883934021        │
└───────────────────────────────┴───────────────────────────────┘

For key:val/fused/mrr_ecg on 2 baselines mean: 0.7936477065086365 std: 0.008189678192138672
For key:val/fused/mrr_vid on 2 baselines mean: 0.6929045915603638 std: 0.0032306313514709473
For key:val/fused/mrr_txt on 2 baselines mean: 0.47781889140605927 std: 0.022247806191444397
For key:val/fused/mrr_eeg on 2 baselines mean: 0.9512632191181183 std: 0.0003866255283355713
For key:val/fused/mrr_mean on 2 baselines mean: 0.7289085984230042 std: 0.006898343563079834
For seed=150
ablated_res for key: val/fused/mrr_ecg  is: 0.7967884540557861
key: val/fused/mrr_ecg  has gain of: 0.003140747547149658 

ablated_res for key: val/fused/mrr_vid  is: 0.7019438743591309
key: val/fused/mrr_vid  has gain of: 0.00903928279876709 

ablated_res for key: val/fused/mrr_txt  is: 0.43402549624443054
key: val/fused/mrr_txt  has gain of: -0.04379339516162872 

ablated_res for key: val/fused/mrr_eeg  is: 0.9623293280601501
key: val/fused/mrr_eeg  has gain of: 0.01106610894203186 

ablated_res for key: val/fused/m

# ECG

In [18]:
mrr_check_keys = [
    # 'val/fused/mrr_ecg',
    'val/fused/mrr_aud',
    'val/fused/mrr_vid',
    'val/fused/mrr_txt',
    'val/fused/mrr_eeg',
    'val/fused/mrr_mean',
]

In [19]:
import numpy as np

# I only care for MRR + meanR@
print("Baseline variance calculations")

# Without text mean
baseline_res[0]["val/fused/mrr_mean"] = 0.8213
baseline_res[0]["val/fused/mrr_ecg"] = 0.6902
baseline_res[0]["val/fused/mrr_aud"] = 0.9383
baseline_res[0]["val/fused/mrr_eeg"] = 0.9641
baseline_res[0]["val/fused/mrr_vid"] = 0.6925
baseline_res[0]["val/fused/mrr_txt"] = 0.5746

for bb in baseline_res:
    bb["val/fused/mrr_mean"] = (
            (bb["val/fused/mrr_vid"] + bb["val/fused/mrr_aud"] + bb["val/fused/mrr_txt"] + bb["val/fused/mrr_eeg"]) / 4
    )

baseline_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_res:
        values.append(res[key])

    arr = np.array(values)
    baseline_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_res)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

Baseline variance calculations
For key:val/fused/mrr_aud on 3 baselines mean: 0.9368384037971497 std: 0.0013239420559275343
For key:val/fused/mrr_vid on 3 baselines mean: 0.6822628990809122 std: 0.007717570297504635
For key:val/fused/mrr_txt on 3 baselines mean: 0.5238234959761302 std: 0.039937979601370535
For key:val/fused/mrr_eeg on 3 baselines mean: 0.9529783913612366 std: 0.007937738596031712
For key:val/fused/mrr_mean on 3 baselines mean: 0.7739757975538571 std: 0.013404469653951381


In [20]:
ecg_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-ecg/2026-03-26_03-59-29/checkpoints/epochepoch=39-stepstep=102120.ckpt"

In [21]:
# Create model instance
inference_ckpt = get_model_ckpt(ecg_ablate_ckpt_150)
mod_less_model = Factory.best_inference(disabled_supports={'ecg'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()
# todo 3 seeds
# And its according datamodule
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

# Now validate it
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

# I only care for MRR at the moment
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.14173243939876556      │
│    val/fused/alignment_eeg    │      0.14672204852104187      │
│    val/fused/alignment_txt    │      0.08447915315628052      │
│    val/fused/alignment_vid    │      0.15206420421600342      │
│     val/fused/margin_aud      │      0.2973160743713379       │
│     val/fused/margin_eeg      │      0.2971594035625458       │
│     val/fused/margin_txt      │      0.3302488625049591       │
│     val/fused/margin_vid      │      0.3810228407382965       │
│ val/fused/meanR@1-3-5-10_aud  │       0.965057909488678       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9774631857872009       │
│ val/fused/meanR@1-3-5-10_mean │      0.8218640089035034       │
│ val/fused/meanR@1-3-5-10_txt  │       0.525501012802124       │
│ val/fused/meanR@1-3-5-10_vid  │      0.8194341063499451       │
│       val/fused/mrr_aud       │      0.9450741410255432       │
│       val/fused/mrr_eeg       │      0.9677215814590454       │
│      val/fused/mrr_mean       │      0.7858515381813049       │
│       val/fused/mrr_txt       │      0.49598851799964905      │
│       val/fused/mrr_vid       │      0.7346218228340149       │
│      val/fused/top10_aud      │      0.9920828938484192       │
│      val/fused/top10_eeg      │      0.9921932220458984       │
│     val/fused/top10_mean      │       0.87492436170578        │
│      val/fused/top10_txt      │      0.5737361907958984       │
│      val/fused/top10_vid      │       0.941685140132904       │
│      val/fused/top1_aud       │      0.9154994487762451       │
│      val/fused/top1_eeg       │      0.9531596302986145       │
│      val/fused/top1_mean      │      0.7320142388343811       │
│      val/fused/top1_txt       │      0.4458570182323456       │
│      val/fused/top1_vid       │      0.6135408878326416       │
│      val/fused/top3_aud       │      0.9687880873680115       │
│      val/fused/top3_eeg       │      0.9783876538276672       │
│      val/fused/top3_mean      │      0.8264681696891785       │
│      val/fused/top3_txt       │      0.5305114984512329       │
│      val/fused/top3_vid       │      0.8281854391098022       │
│      val/fused/top5_aud       │      0.9838612079620361       │
│      val/fused/top5_eeg       │      0.9861122369766235       │
│      val/fused/top5_mean      │      0.8540494441986084       │
│      val/fused/top5_txt       │      0.5518994927406311       │
│      val/fused/top5_vid       │      0.8943249583244324       │
│        val/fusion-loss        │      2.9819912910461426       │
│        val/fusion/aud         │      2.9096338748931885       │
│        val/fusion/eeg         │      2.8396191596984863       │
│        val/fusion/txt         │      3.5012035369873047       │
│        val/fusion/vid         │        2.8634033203125        │
│           val/loss            │      2.9819912910461426       │
└───────────────────────────────┴───────────────────────────────┘

For seed=150
ablated_res for key: val/fused/mrr_aud  is: 0.9450741410255432
key: val/fused/mrr_aud  has gain of: 0.008235737228393547 

ablated_res for key: val/fused/mrr_vid  is: 0.7346218228340149
key: val/fused/mrr_vid  has gain of: 0.05235892375310269 

ablated_res for key: val/fused/mrr_txt  is: 0.49598851799964905
key: val/fused/mrr_txt  has gain of: -0.027834977976481157 

ablated_res for key: val/fused/mrr_eeg  is: 0.9677215814590454
key: val/fused/mrr_eeg  has gain of: 0.014743190097808778 

ablated_res for key: val/fused/mrr_mean  is: 0.7858515381813049
key: val/fused/mrr_mean  has gain of: 0.011875740627447784 



## Baseline without mod

In [22]:
baseline_mod_less_results = []
for b in baselines[1:]:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

baseline_mod_less_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_mod_less_results:
        values.append(res[key])

    arr = np.array(values)
    baseline_mod_less_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_mod_less_results)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

# Compare now
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_mod_less_metrics[key], "\n")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.13900324702262878      │
│    val/fused/alignment_eeg    │      0.14375069737434387      │
│    val/fused/alignment_txt    │      0.08423597365617752      │
│    val/fused/alignment_vid    │      0.15779626369476318      │
│     val/fused/margin_aud      │      0.30008938908576965      │
│     val/fused/margin_eeg      │      0.2938310205936432       │
│     val/fused/margin_txt      │      0.3167465031147003       │
│     val/fused/margin_vid      │      0.38876378536224365      │
│ val/fused/meanR@1-3-5-10_aud  │      0.9668087959289551       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9646026492118835       │
│ val/fused/meanR@1-3-5-10_mean │      0.8219945430755615       │
│ val/fused/meanR@1-3-5-10_txt  │      0.5482724905014038       │
│ val/fused/meanR@1-3-5-10_vid  │      0.8082941770553589       │
│       val/fused/mrr_aud       │       0.94907146692276        │
│       val/fused/mrr_eeg       │      0.9497940540313721       │
│      val/fused/mrr_mean       │      0.7851811647415161       │
│       val/fused/mrr_txt       │       0.522983193397522       │
│       val/fused/mrr_vid       │      0.7188757658004761       │
│      val/fused/top10_aud      │      0.9922351241111755       │
│      val/fused/top10_eeg      │      0.9866874814033508       │
│     val/fused/top10_mean      │      0.8766727447509766       │
│      val/fused/top10_txt      │       0.591684103012085       │
│      val/fused/top10_vid      │      0.9360843300819397       │
│      val/fused/top1_aud       │      0.9232643842697144       │
│      val/fused/top1_eeg       │      0.9284246563911438       │
│      val/fused/top1_mean      │      0.7306007742881775       │
│      val/fused/top1_txt       │      0.47965899109840393      │
│      val/fused/top1_vid       │      0.5910550951957703       │
│      val/fused/top3_aud       │      0.9695493578910828       │
│      val/fused/top3_eeg       │      0.9664721488952637       │
│      val/fused/top3_mean      │       0.827339768409729       │
│      val/fused/top3_txt       │      0.5526473522186279       │
│      val/fused/top3_vid       │      0.8206902146339417       │
│      val/fused/top5_aud       │      0.9821863770484924       │
│      val/fused/top5_eeg       │      0.9768263101577759       │
│      val/fused/top5_mean      │      0.8533648252487183       │
│      val/fused/top5_txt       │      0.5690996050834656       │
│      val/fused/top5_vid       │      0.8853471279144287       │
│        val/fusion-loss        │       2.967712640762329       │
│        val/fusion/aud         │      2.9361143112182617       │
│        val/fusion/eeg         │      2.8789403438568115       │
│        val/fusion/txt         │      3.4694786071777344       │
│        val/fusion/vid         │       2.791501522064209       │
│           val/loss            │       2.967712640762329       │
└───────────────────────────────┴───────────────────────────────┘

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Validate metric        ┃         DataLoader 0          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│    val/fused/alignment_aud    │      0.1423906534910202       │
│    val/fused/alignment_eeg    │      0.14331106841564178      │
│    val/fused/alignment_txt    │      0.08450128138065338      │
│    val/fused/alignment_vid    │      0.15980054438114166      │
│     val/fused/margin_aud      │      0.29981547594070435      │
│     val/fused/margin_eeg      │      0.2907319664955139       │
│     val/fused/margin_txt      │      0.3304195702075958       │
│     val/fused/margin_vid      │      0.3802868127822876       │
│ val/fused/meanR@1-3-5-10_aud  │      0.9697777628898621       │
│ val/fused/meanR@1-3-5-10_eeg  │      0.9650956988334656       │
│ val/fused/meanR@1-3-5-10_mean │      0.8156352639198303       │
│ val/fused/meanR@1-3-5-10_txt  │      0.5156670808792114       │
│ val/fused/meanR@1-3-5-10_vid  │      0.8120006322860718       │
│       val/fused/mrr_aud       │      0.9512428641319275       │
│       val/fused/mrr_eeg       │      0.9522674083709717       │
│      val/fused/mrr_mean       │      0.7761980295181274       │
│       val/fused/mrr_txt       │      0.4758947193622589       │
│       val/fused/mrr_vid       │      0.7253871560096741       │
│      val/fused/top10_aud      │      0.9942144155502319       │
│      val/fused/top10_eeg      │      0.9862765669822693       │
│     val/fused/top10_mean      │      0.8739087581634521       │
│      val/fused/top10_txt      │       0.579718828201294       │
│      val/fused/top10_vid      │      0.9354254007339478       │
│      val/fused/top1_aud       │       0.923568844795227       │
│      val/fused/top1_eeg       │      0.9338482618331909       │
│      val/fused/top1_mean      │      0.7178032398223877       │
│      val/fused/top1_txt       │      0.41145676374435425      │
│      val/fused/top1_vid       │      0.6023391485214233       │
│      val/fused/top3_aud       │      0.9759440422058105       │
│      val/fused/top3_eeg       │      0.9652395248413086       │
│      val/fused/top3_mean      │      0.8204637765884399       │
│      val/fused/top3_txt       │      0.5206401348114014       │
│      val/fused/top3_vid       │      0.8200312852859497       │
│      val/fused/top5_aud       │      0.9853837490081787       │
│      val/fused/top5_eeg       │      0.9750184416770935       │
│      val/fused/top5_mean      │      0.8503653407096863       │
│      val/fused/top5_txt       │      0.5508525371551514       │
│      val/fused/top5_vid       │      0.8902066946029663       │
│        val/fusion-loss        │       2.974384307861328       │
│        val/fusion/aud         │      2.9127190113067627       │
│        val/fusion/eeg         │      2.8865652084350586       │
│        val/fusion/txt         │      3.4700653553009033       │
│        val/fusion/vid         │       2.792057752609253       │
│           val/loss            │       2.974384307861328       │
└───────────────────────────────┴───────────────────────────────┘

For key:val/fused/mrr_aud on 2 baselines mean: 0.9501571655273438 std: 0.0010856986045837402
For key:val/fused/mrr_vid on 2 baselines mean: 0.7221314609050751 std: 0.003255695104598999
For key:val/fused/mrr_txt on 2 baselines mean: 0.49943895637989044 std: 0.02354423701763153
For key:val/fused/mrr_eeg on 2 baselines mean: 0.9510307312011719 std: 0.0012366771697998047
For key:val/fused/mrr_mean on 2 baselines mean: 0.7806895971298218 std: 0.004491567611694336
For seed=150
ablated_res for key: val/fused/mrr_aud  is: 0.9450741410255432
key: val/fused/mrr_aud  has gain of: -0.005083024501800537 

ablated_res for key: val/fused/mrr_vid  is: 0.7346218228340149
key: val/fused/mrr_vid  has gain of: 0.01249036192893982 

ablated_res for key: val/fused/mrr_txt  is: 0.49598851799964905
key: val/fused/mrr_txt  has gain of: -0.003450438380241394 

ablated_res for key: val/fused/mrr_eeg  is: 0.9677215814590454
key: val/fused/mrr_eeg  has gain of: 0.016690850257873535 

ablated_res for key: val/fused

In [ ]:
# Need more ECG and Aud seed, still in noise atm